In [1]:
import numpy as np
import trimesh

INPUT_MESH = "/Users/jacob/Desktop/Uni/7_semester/Bachelor/Meshes/final_015_C2_BEAG.obj"

mesh = trimesh.load(
    INPUT_MESH,
    process=False
)

print(f"faces: {mesh.faces.shape}")
print(f"vertices: {mesh.vertices.shape}")
print(f"colors: {mesh.visual.vertex_colors.shape}")

faces: (539088, 3)
vertices: (269548, 3)
colors: (269548, 4)


In [2]:
from pathlib import Path

import numpy as np
import trimesh

# ============================================================
# SETTINGS
# ============================================================

MESH_DIR = Path("/Users/jacob/Desktop/Uni/7_semester/Bachelor/Downsampled_meshes")
AREA_EPSILON = 1e-12  # Absolute threshold in squared mesh-coordinate units


def check_obj(path: Path) -> None:
    print(f"\nChecking: {path.name}")

    try:
        mesh = trimesh.load(path, process=False, force="mesh")
    except Exception as exc:
        print(f"  LOAD FAILED: {exc}")
        return

    vertices = np.asarray(mesh.vertices)
    faces = np.asarray(mesh.faces)

    print(f"  Vertices: {len(vertices):,}")
    print(f"  Faces:    {len(faces):,}")

    if vertices.ndim != 2 or vertices.shape[1] != 3:
        print(f"  ERROR: expected vertices shaped (N, 3), got {vertices.shape}")
        return

    if faces.ndim != 2 or faces.shape[1] != 3:
        print(f"  ERROR: expected triangular faces shaped (M, 3), got {faces.shape}")
        return

    nonfinite_vertices = ~np.isfinite(vertices).all(axis=1)
    print(f"  Vertices with non-finite coordinates: {nonfinite_vertices.sum():,}")

    invalid_indices = (faces < 0) | (faces >= len(vertices))
    invalid_faces = invalid_indices.any(axis=1)
    print(f"  Faces with invalid vertex indices: {invalid_faces.sum():,}")

    # Only calculate geometry for faces whose indices are valid.
    valid_faces = faces[~invalid_faces]

    repeated_index = (
        (valid_faces[:, 0] == valid_faces[:, 1])
        | (valid_faces[:, 1] == valid_faces[:, 2])
        | (valid_faces[:, 2] == valid_faces[:, 0])
    )
    print(f"  Faces with repeated indices: {repeated_index.sum():,}")

    tri = vertices[valid_faces]
    double_area = np.linalg.norm(
        np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0]),
        axis=1,
    )
    bad_local_indices = np.flatnonzero(double_area <= AREA_EPSILON)
    bad_faces = valid_faces[bad_local_indices]

    print("  First bad face indices:", bad_local_indices[:10])
    print("  First bad vertex triples:", bad_faces[:10])
    print("  First bad triangles:\n", vertices[bad_faces[:10]])
    
    zero_area = double_area <= AREA_EPSILON
    print(f"  Faces with double-area <= {AREA_EPSILON:g}: {zero_area.sum():,}")

    # Detect duplicate triangles regardless of vertex ordering.
    sorted_faces = np.sort(valid_faces, axis=1)
    _, first_indices = np.unique(sorted_faces, axis=0, return_index=True)
    duplicate_faces = np.ones(len(valid_faces), dtype=bool)
    duplicate_faces[first_indices] = False
    print(f"  Duplicate faces: {duplicate_faces.sum():,}")

    if len(double_area):
        print(f"  Minimum double-area: {double_area.min():.6g}")
        print(f"  Median double-area:  {np.median(double_area):.6g}")
        print(f"  Maximum double-area: {double_area.max():.6g}")


# ============================================================
# CHECK ALL OBJ FILES
# ============================================================

obj_files = sorted(MESH_DIR.glob("*.obj"))

if not obj_files:
    print(f"No OBJ files found in {MESH_DIR}")
else:
    for obj_path in obj_files:
        check_obj(obj_path)


Checking: brain_segmented_colored_02.obj
  Vertices: 49,992
  Faces:    100,000
  Vertices with non-finite coordinates: 0
  Faces with invalid vertex indices: 0
  Faces with repeated indices: 0
  First bad face indices: []
  First bad vertex triples: []
  First bad triangles:
 []
  Faces with double-area <= 1e-12: 0
  Duplicate faces: 0
  Minimum double-area: 0.0023462
  Median double-area:  0.24333
  Maximum double-area: 3.45056

Checking: brain_segmented_grey_02.obj
  Vertices: 49,992
  Faces:    100,000
  Vertices with non-finite coordinates: 0
  Faces with invalid vertex indices: 0
  Faces with repeated indices: 0
  First bad face indices: []
  First bad vertex triples: []
  First bad triangles:
 []
  Faces with double-area <= 1e-12: 0
  Duplicate faces: 0
  Minimum double-area: 0.0023462
  Median double-area:  0.24333
  Maximum double-area: 3.45056

Checking: final_002_A2_AFGH.obj
  Vertices: 50,004
  Faces:    99,991
  Vertices with non-finite coordinates: 0
  Faces with invalid